# DS 5001 Final Project

Erin Siedlecki

## Parsed and Annotated Data

In [1]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

np.random.seed(42)

csv_path = Path("../Song Data/csv")
dfs = []

for file in csv_path.glob("*.csv"):
        if file.name.lower() in ['bts.csv', 'eminem.csv', 'drake.csv', 'nickiminaj.csv', 'cardib.csv']:
            continue
        temp_data = pd.read_csv(file)
        temp_data = temp_data.drop(columns=['Unnamed: 0', '#'], errors='ignore')
        dfs.append(temp_data)

df = pd.concat(dfs, ignore_index=True)
df = df.rename(columns={
    'Artist': 'artist',
    'Title': 'song',
    'Album': 'album',
    'Year': 'year',
    'Date': 'date',
    'Lyric': 'lyrics'
})

artifact_note =  r"lyrics for this song have yet to be released please check back once the song has been released"
df = df[~df['lyrics'].str.lower().str.contains(artifact_note, na=False)].copy()
df = df[~df['lyrics'].str.lower().str.contains(r'\btba\b', na=False)].copy()
df.head()

,artist,song,album,year,date,lyrics
0,Dua Lipa,New Rules,Dua Lipa,2017.0,2017-06-02,one one one one one talkin' in my sleep at n...
1,Dua Lipa,Don’t Start Now,Future Nostalgia,2019.0,2019-11-01,if you don't wanna see me did a full 80 craz...
2,Dua Lipa,IDGAF,Dua Lipa,2017.0,2017-06-02,you call me all friendly tellin' me how much y...
3,Dua Lipa,Blow Your Mind (Mwah),Dua Lipa,2016.0,2016-08-26,i know it's hot i know we've got something tha...
4,Dua Lipa,Be the One,Dua Lipa,2015.0,2015-10-30,i see the moon i see the moon i see the moon o...


In [2]:
df.isna().sum()

artist       0
song         0
album     1015
year      1127
date      1127
lyrics      34
dtype: int64

In [3]:
df = df.dropna(subset=['lyrics'])
df['album'] = df['album'].fillna('Unknown')
df['year'] = df['year'].fillna(-1)
df['year'] = df['year'].astype(int)
df['date'] = df['date'].fillna('Unknown')

In [4]:
df.isna().sum()

artist    0
song      0
album     0
year      0
date      0
lyrics    0
dtype: int64

In [5]:
df.shape[0]

4114

In [6]:
print(df['lyrics'].iloc[0])

one one one one one   talkin' in my sleep at night makin' myself crazy out of my mind out of my mind wrote it down and read it out hopin' it would save me too many times too many times  refrain my love he makes me feel like nobody else nobody else but my love he doesn't love me so i tell myself i tell myself  pre one don't pick up the phone you know he's only callin' 'cause he's drunk and alone two don't let him in you'll have to kick him out again three don't be his friend you know you're gonna wake up in his bed in the morning and if you're under him you ain't gettin' over him   i got new rules i count 'em i got new rules i count 'em i gotta tell them to myself i got new rules i count 'em i gotta tell them to myself   i keep pushin' forwards but he keeps pullin' me backwards nowhere to turn no way nowhere to turn no now i'm standin' back from it i finally see the pattern i never learn i never learn  refrain but my love he doesn't love me so i tell myself i tell myself i do i do i do 

## LIB

In [7]:
import matplotlib.pyplot as plt
import seaborn as sns
from glob import glob
import re
import nltk

In [8]:
LIB = df.copy()
LIB['doc_length_words'] = LIB['lyrics'].str.split().str.len()
LIB['doc_length_chars'] = LIB['lyrics'].str.len()
LIB = LIB.drop_duplicates(subset=['artist', 'song', 'album']).reset_index(drop=True)
LIB = LIB.drop(columns=['lyrics'])
avg_length = LIB['doc_length_chars'].mean()
print(round(avg_length, 2))

1646.25


In [9]:
LIB

,artist,song,album,year,date,doc_length_words,doc_length_chars
0,Dua Lipa,New Rules,Dua Lipa,2017,2017-06-02,486,2329
1,Dua Lipa,Don’t Start Now,Future Nostalgia,2019,2019-11-01,307,1478
2,Dua Lipa,IDGAF,Dua Lipa,2017,2017-06-02,445,2027
3,Dua Lipa,Blow Your Mind (Mwah),Dua Lipa,2016,2016-08-26,429,2082
4,Dua Lipa,Be the One,Dua Lipa,2015,2015-10-30,406,1700
...,...,...,...,...,...,...,...
4109,Khalid,Young dumb,Unknown,2017,2017-02-02,145,758
4110,Khalid,Khalid - Vertigo (Tradução Português),Unknown,2018,2018-10-28,298,1765
4111,Khalid,Better (Miles Away Remix),Unknown,2018,2018-12-12,322,1609
4112,Khalid,Khalid - Better (Official Music Video),Unknown,2018,2018-05-07,243,1459


Number of observations: 4,114

List of features: artist, song, album, year, date, doc_length_words, doc_length_chars

Average length of each document in characters: 1,646.25

In [10]:
LIB.to_csv('../data/LIB.csv', sep='|', index=False)

## CORPUS

In [11]:
import re
import nltk

nltk.download('punkt')
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/erinsiedlecki/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /Users/erinsiedlecki/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


True

In [12]:
custom_stopwords = {'lyrics', 'lyric', 'snippet', 'unreleased', 
                    'released', 're', 'm', 'don', 't', 'll', 
                    'la', 'song', 'available'}

CORPUS = df.copy()
def tokenize(text):
    return re.findall(r"\b\w+\b", str(text).lower())
CORPUS['token_str'] = CORPUS['lyrics'].apply(tokenize)
CORPUS = CORPUS.explode('token_str').reset_index(drop=True)
CORPUS = CORPUS.dropna(subset=['token_str'])

# filter out non-english tokens
CORPUS = CORPUS[CORPUS['token_str'].apply(lambda x: x.isascii())].copy()

CORPUS['term_str'] = CORPUS['token_str']

# remove custom stopwords
CORPUS = CORPUS[~CORPUS['term_str'].isin(custom_stopwords)].copy()

CORPUS['token_num'] = CORPUS.groupby(['artist', 'song']).cumcount()
CORPUS['pos'] = [tag for _, tag in nltk.pos_tag(CORPUS['token_str'].tolist())]
def pos_group(tag):
    if tag.startswith('NN'):
        return 'NOUN'
    elif tag.startswith('VB'):
        return 'VERB'
    elif tag.startswith('JJ'):
        return 'ADJ'
    elif tag.startswith('RB'):
        return 'ADV'
    else:
        return 'OTHER'
CORPUS['pos_group'] = CORPUS['pos'].apply(pos_group)
CORPUS = CORPUS.drop(columns=['lyrics'])
OHCO = ['artist', 'song', 'token_num']
CORPUS = CORPUS.set_index(OHCO)
CORPUS

album  year        date  \
artist   song                   token_num                               
Dua Lipa New Rules              0          Dua Lipa  2017  2017-06-02   
                                1          Dua Lipa  2017  2017-06-02   
                                2          Dua Lipa  2017  2017-06-02   
                                3          Dua Lipa  2017  2017-06-02   
                                4          Dua Lipa  2017  2017-06-02   
...                                             ...   ...         ...   
Khalid   Better (Rennie! Remix) 304         Unknown  2019  2019-02-01   
                                305         Unknown  2019  2019-02-01   
                                306         Unknown  2019  2019-02-01   
                                307         Unknown  2019  2019-02-01   
                                308         Unknown  2019  2019-02-01   

                                          token_str term_str  pos pos_group  
artist   song                   token_num                                    
Dua Lipa New Rules              0               one      one   CD     OTHER  
                                1               one      one   CD     OTHER  
                                2               one      one   CD     OTHER  
                                3               one      one   CD     OTHER  
                                4               one      one   CD     OTHER  
...                                             ...      ...  ...       ...  
Khalid   Better (Rennie! Remix) 304         nothing  nothing   NN      NOUN  
                                305           feels    feels  NNS      NOUN  
                                306          better   better  RBR       ADV  
                                307            than     than   IN     OTHER  
                                308            this     this   DT     OTHER  

[1390794 rows x 7 columns]

Number of observations (should be >= 500,000 and <= 2,000,000 observations): 1,390,794

OHCO Structure: artist, song, token_num

Columns: album, year, date, token_str, term_str, pos, pos_group

In [13]:
CORPUS.to_csv('../data/CORPUS.csv', sep='|', index=True)

## VOCAB

In [14]:
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/erinsiedlecki/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [15]:
VOCAB = CORPUS.term_str.value_counts().to_frame('n').sort_index()
VOCAB.index.name = 'term_str'
VOCAB['n_chars'] = VOCAB.index.str.len()
VOCAB['p'] = VOCAB.n / VOCAB.n.sum()
VOCAB['i'] = -np.log2(VOCAB.p)
VOCAB['max_pos'] = CORPUS[['term_str','pos']].value_counts().unstack(fill_value=0).idxmax(1)
VOCAB['max_pos_group'] = CORPUS[['term_str','pos_group']].value_counts().unstack(fill_value=0).idxmax(1)
sw = set(nltk.corpus.stopwords.words('english'))
VOCAB['stop'] = VOCAB.index.isin(sw)
stemmer1 = PorterStemmer()
VOCAB['porter_stem'] = VOCAB.apply(lambda x: stemmer1.stem(x.name), 1)
VOCAB

,n,n_chars,p,i,max_pos,max_pos_group,stop,porter_stem
term_str,,,,,,,,
0,185,1,1.330175e-04,12.876096,CD,OTHER,False,0
00,41,2,2.947956e-05,15.049925,CD,OTHER,False,00
000,14,3,1.006619e-05,16.600122,CD,OTHER,False,000
00000,1,5,7.190137e-07,20.407477,CD,OTHER,False,00000
000s,1,4,7.190137e-07,20.407477,CD,OTHER,False,000
...,...,...,...,...,...,...,...,...
zurich,1,6,7.190137e-07,20.407477,JJ,ADJ,False,zurich
zuse,2,4,1.438027e-06,19.407477,CD,NOUN,False,zuse
zuzu,1,4,7.190137e-07,20.407477,NN,NOUN,False,zuzu


Number of observations: 20,192

Columns: n, n_chars, p, i, max_pos, max_pos_group, stop, porter_stem, df, idf, dfidf, tfidf_mean

Top 20 significant words in the corpus by DFIDF: ['let', 'time', 'one', 'see', 'say','never', 'baby', 'get', 'yeah', 'pre', 'want', 'go', 'got', 'make', 'way', 'wanna', 'take', 'back', 'come', 'need']

In [16]:
VOCAB.to_csv('../data/VOCAB_base.csv', sep='|', index=True)